In [8]:

import numpy as np
import pandas as pd
import json
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from multiprocessing import Pool
import warnings
warnings.filterwarnings("ignore")


heroes = pd.read_csv('data_heroes.csv')
objects = pd.read_csv('data_objects.csv')


dis = pd.read_csv('dist_objects.csv')
d0 = pd.read_csv('dist_start.csv')
d0['end'] = 0
d0.index = range(0,len(d0))
d0 = d0.iloc[:,1:]
dis = pd.concat([d0,dis],axis=1)

d0 = d0.T
d0 = pd.concat([d0.iloc[:,:2],d0], axis =1)
d0.iloc[:,:2] = 0
d0.columns = dis.columns
dis = pd.concat([d0,dis],axis=0)
dis.columns = [0,-1]+list(np.arange(700)+1)
dis.index = dis.columns

heroes = heroes.iloc[:20]


dis = dis.loc[[0,-1]+[-1]*6*20+list(np.arange(len(dis)-2)+1),[0,-1]+[-1]*6*20+list(np.arange(len(dis)-2)+1)]
dis

,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,691,692,693,694,695,696,697,698,699,700
0,0,0,0,0,0,0,0,0,0,0,...,426,493,453,426,666,720,693,600,480,120
-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696,720,0,0,0,0,0,0,0,0,0,...,567,600,647,760,240,0,233,187,747,853
697,693,0,0,0,0,0,0,0,0,0,...,527,553,573,700,233,233,0,160,653,800
698,600,0,0,0,0,0,0,0,0,0,...,467,500,520,653,80,187,160,0,647,747
699,480,0,0,0,0,0,0,0,0,0,...,573,573,267,680,693,747,653,647,0,587


In [2]:
objects = objects.loc[[1,1]+list(objects.index)]
objects.index=np.arange(len(objects))-1
objects.loc[:,'reward']=100 

objects.iloc[:2]=1
objects = objects.loc[dis.index]
objects['start'] = (objects['day_open']-1)*24*60
objects['finish'] =(objects['day_open'])*24*60
objects['finish'].iloc[:2] = 7*24*60
objects.loc[[0,-1],'reward'] = 0
objects['day_open'].iloc[2:122] = [1,2,3,4,5,6]*20
objects['start'].iloc[2:2+6*20] = objects['day_open'].iloc[2:2+6*20]*24*60-1
objects['finish'].iloc[2:2+6*20] =objects['day_open'].iloc[2:2+6*20]*24*60+1
objects

,object_id,day_open,reward,start,finish
0,1,1,0,0,10080
-1,1,1,0,0,10080
-1,1,1,0,1439,1441
-1,1,2,0,2879,2881
-1,1,3,0,4319,4321
...,...,...,...,...,...
696,696,6,100,7200,8640
697,697,4,100,4320,5760
698,698,6,100,7200,8640
699,699,3,100,2880,4320


In [ ]:
dis = dis.iloc[:200,:200]
objects = objects.iloc[:200,:]


In [4]:


data = {}
data['time_matrix'] = dis.values.tolist()
data['time_windows'] = objects[['start','finish']].astype(int).values.tolist()
data['demands'] = objects['reward'].to_list()

data['num_vehicles'] = len(heroes['move_points'].to_list())
data['vehicle_capacities'] = heroes['move_points'].to_list()
data['num'] = len(data['time_matrix'])



In [ ]:
"""Vehicles Routing Problem (VRP) with Time Windows and Capacity Constraints."""

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def export_solution(data, manager, routing, solution):
    """Exports solution to a JSON file."""
    time_dimension = routing.GetDimensionOrDie("Time")
    capacity_dimension = routing.GetDimensionOrDie("Capacity")
    
    routes_data = []
    total_time = 0
    alst = []
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        route_stops = []
        route_times = []
        route_loads = []
        route_slack=[]
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            time_var = time_dimension.CumulVar(index)
            load_var = capacity_dimension.CumulVar(index)
            
            route_stops.append(node_index)
            # Store the specific value assigned by the solver
            route_times.append(solution.Value(time_var)) 
            route_loads.append(solution.Value(load_var))
    
            index = solution.Value(routing.NextVar(index))
        
        # End node (depot)
        node_index = manager.IndexToNode(index)
        time_var = time_dimension.CumulVar(index)
        load_var = capacity_dimension.CumulVar(index)
        route_stops.append(node_index)
        route_times.append(solution.Value(time_var))
        route_loads.append(solution.Value(load_var))
        
        route_time = solution.Value(time_var)
        total_time += route_time

        if len(route_stops)>2:
            routes_data.append({
                "vehicle_id": vehicle_id,
                'move_point':heroes['move_points'].to_list()[vehicle_id],
                "stops": route_stops,
                'days':[objects['day_open'].to_list()[i] for i in route_stops],
                "arrival_times": route_times,
                "loads": route_loads,
                "route_duration": route_time
            })
        alst += route_stops
    print(f'Drop stops {len(set(range(data['num']))-set(alst))}')
    return routes_data


# Create the routing index manager.
manager = pywrapcp.RoutingIndexManager(
    len(data["time_matrix"]), data["num_vehicles"], [0]*data['num_vehicles'],[1]*data['num_vehicles']
)

# Create Routing Model.
routing = pywrapcp.RoutingModel(manager)

# Create and register a transit callback.
def time_callback(from_index, to_index):
    """Returns the travel time between the two nodes."""
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data["time_matrix"][from_node][to_node]+3

transit_callback_index = routing.RegisterTransitCallback(time_callback)

# Define cost of each arc.
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Add Time Windows constraint.
time = "Time"
routing.AddDimension(
    transit_callback_index,
    30000,  # allow waiting time
    7*24*60,  # maximum time per vehicle
    True,  # Don't force start cumul to zero.
    time,
)
time_dimension = routing.GetDimensionOrDie(time)
# Add time window constraints for each location except depot.
for location_idx, time_window in enumerate(data["time_windows"]):
    if location_idx in [0,1]:
        continue
    index = manager.NodeToIndex(location_idx)
    time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])
# Add time window constraints for each vehicle start node.
for vehicle_id in range(data["num_vehicles"]):
    index = routing.Start(vehicle_id)
    time_dimension.CumulVar(index).SetRange(
        data["time_windows"][0][0], data["time_windows"][0][1]
    )

# Instantiate route start and end times to produce feasible times.
for i in range(data["num_vehicles"]):
    routing.AddVariableMinimizedByFinalizer(
        time_dimension.CumulVar(routing.Start(i))
    )

# --- NEW: Add Capacity Constraint ---
# Create and register a demand callback.
def demand_callback(from_index, to_index):
    """Returns the travel time between the two nodes."""
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data["time_matrix"][from_node][to_node]+data['demands'][from_node]

demand_callback_index = routing.RegisterTransitCallback(demand_callback)

# Add Capacity constraint to the model.
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    12000,  # null capacity slack
    data["vehicle_capacities"],  # vehicle maximum capacities
    True,  # start cumul to zero
    "Capacity",
)

solver = routing.solver()

for i in range(data['num']):
    index = manager.IndexToNode(i)
    cumul = time_dimension.CumulVar(index)
    slack = time_dimension.SlackVar(index)
    #cumul.SetRange(0, 1000*(slack==0))

cap_dimension = routing.GetDimensionOrDie("Capacity")
for node in range(2, 122):
    index = manager.IndexToNode(node)
    #solver.Add(routing.VehicleVar(index)==int((node-2)//7))


for node in range(2, len(data['time_matrix'])):
    index = manager.IndexToNode(node)
    routing.AddDisjunction([index], 100000)
    #cap_dimension.SlackVar(index).SetRange(0,200)


# Setting first solution heuristic.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

# Solve the problem.
solution = routing.SolveWithParameters(search_parameters)

print('Done')
# Print solution on console.
if solution:
    routes_data=export_solution(data, manager, routing, solution)
else:
    print("No solution found!")



In [ ]:
submit_csv = pd.DataFrame()
for i in routes_data:

    pf = objects.iloc[i['stops'][1:-1]]
    pf['hero_id'] = heroes.iloc[i['vehicle_id']]['hero_id']
    pf['num']= range(len(pf))
    pf['move'] = i['move_point']
    pf['windows'] = i['arrival_times'][1:-1]
    submit_csv = pd.concat([submit_csv, pf])

submit_csv = submit_csv.sort_values(['hero_id', 'num'])   

submit_csv

,object_id,day_open,reward,start,finish,hero_id,num,move,windows
17,17,1,100,0,1440,1,0,1700,283
44,44,1,100,0,1440,1,1,1700,893
60,60,1,100,0,1440,1,2,1700,996
-1,1,1,0,1439,1441,1,3,1700,1439
23,23,2,100,1440,2880,1,4,1700,1442
...,...,...,...,...,...,...,...,...,...
-1,1,3,0,4319,4321,20,6,1560,4319
-1,1,4,0,5759,5761,20,7,1560,5759
-1,1,5,0,7199,7201,20,8,1560,7199
-1,1,6,0,8639,8641,20,9,1560,8639


In [ ]:
submit_csv = submit_csv.loc[submit_csv['reward']!=0]
submit_csv[['hero_id','object_id']].to_csv('submit.csv', index=False)


In [ ]:
routes_data

[{'vehicle_id': 0,
  'move_point': 1700,
  'stops': [0, 138, 165, 181, 2, 144, 3, 4, 123, 23, 18, 7, 172, 1],
  'days': [1, 1, 1, 1, 1, 2, 2, 3, 4, 4, 5, 6, 7, 1],
  'arrival_times': [0,
   283,
   893,
   996,
   1439,
   1442,
   2879,
   4319,
   4322,
   5759,
   7199,
   8639,
   8642,
   8645],
  'loads': [0,
   280,
   987,
   1187,
   1287,
   1287,
   1387,
   1387,
   1387,
   1487,
   1487,
   1487,
   1487,
   1587],
  'route_duration': 8645},
 {'vehicle_id': 1,
  'move_point': 1560,
  'stops': [0,
   176,
   178,
   177,
   191,
   145,
   56,
   150,
   51,
   125,
   46,
   124,
   29,
   24,
   13,
   141,
   1],
  'days': [1, 1, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 6, 7, 1],
  'arrival_times': [0,
   349,
   377,
   405,
   555,
   651,
   1439,
   1442,
   2879,
   2882,
   4319,
   4322,
   5759,
   7199,
   8639,
   8642,
   8645],
  'loads': [0,
   346,
   471,
   596,
   843,
   1036,
   1136,
   1136,
   1236,
   1236,
   1336,
   1336,
   1436,
   1436,
   1436,
